# Processing Agent — Code Guide

**Owner:** Aurora Ruci  
**Target file:** `agents/aurora_processing.py`  

> This notebook is documentation only — do not run it. It mirrors the actual agent code block by block and explains what each part does in plain English to make it easier for us to understand the project and our code inside out.

## What this agent does

The Processing Agent is the first step in the pipeline. Its job is to take raw financial news headlines (from the FNSPID dataset) and pair each headline with stock price data from Yahoo Finance. For every headline, it figures out how the stock price moved the day after the article was published, then labels that move as **up**, **down**, or **neutral**.

The output is `processed_data.csv`, which gets handed off to Nadi's Classifier Agent.

### Note on usefulness

This is useful for the team member herself (me, Aurora :)) because of not being super technical so it is useful to have very extensive comments and explanations of what my code does since AI helps with generating it.

## 1. Imports

These are the external libraries the agent needs:

- **`os` / `sys`** — standard Python tools for working with file paths
- **`pickle`** - used to save and reload the price cache so we don't re-download prices every run
- **`pandas`**
- **`yfinance`** — downloads stock price history from Yahoo Finance
- **`langgraph`** — the framework that connects all five agents together into a pipeline
- **`PipelineState`** — the shared state object that passes information between agents (defined in `agents/state.py`)
- **`Agent`** — Jack's base class (defined in `agents/base.py`) that every agent in the team subclasses

In [ ]:
import os
import pickle
import sys
import pandas as pd
import yfinance as yf
from langgraph.graph import StateGraph, START, END

# __file__ is the path to this script itself
# os.path.abspath gives the full absolute path (e.g. /Users/aurora/NLP_lab/agents/aurora_processing.py)
# os.path.dirname strips the filename and gives the folder (e.g. /Users/aurora/NLP_lab/agents)
# calling dirname again goes one level up to the repo root (e.g. /Users/aurora/NLP_lab)
# sys.path.insert(0, ...) adds that repo root to Python's list of places to look for imports
# without this "from agents.state import ..." would fail when running the script directly
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from agents.state import PipelineState  # the shared state object that all agents read from and write to
from agents.base import Agent            # Jack's base class — every agent in the team subclasses this

## 2. Helper — Fetching stock prices

This function downloads closing prices from Yahoo Finance for every ticker (stock symbol) that appears in our news dataset. It loops through each ticker and pulls the full price history for the date range we need.

It handles:
- If a ticker returns no data (e.g. delisted stock), it's logged and skipped
- Timestamps are normalised (time-of-day is stripped) so that dates match cleanly when we join to headlines
- Only the **closing price** column is kept

The result is a dictionary: `{ "AAPL": <Series of dates → prices>, "TSLA": ..., ... }`

In [ ]:
def _fetch_price_cache(tickers, date_min, date_max):
    # tickers: list of stock symbols e.g. ["AAPL", "TSLA", ...]
    # date_min / date_max: the date range to fetch prices for

    cache, failed = {}, []  # cache will hold results; failed tracks tickers with no data

    for ticker in tickers:
        try:
            # ask Yahoo Finance for the full price history for this ticker
            hist = yf.Ticker(ticker).history(start=date_min, end=date_max)

            if hist.empty:
                # Yahoo returned nothing (ticker may be delisted because of old name or misspelled?)
                print(f"  EMPTY  {ticker}")
                failed.append(ticker)
                continue  # skip to the next ticker

            # Yahoo Finance returns timestamps with timezone info (e.g. "2021-03-15 00:00 UTC")
            # we strip the timezone and time-of-day so dates look like plain "2021-03-15"
            # this makes them match the dates in our news data when we join later
            if hist.index.tz is not None:
                hist.index = hist.index.tz_convert(None).normalize()
            else:
                hist.index = hist.index.normalize()

            cache[ticker] = hist["Close"]  # only keep the closing price column

        except Exception as e:
            # something unexpected went wrong (e.g. network error)
            print(f"  FAILED {ticker}: {e}")
            failed.append(ticker)

    print(f"\nCached {len(cache)} / {len(tickers)} tickers. Failed: {failed or 'none'}")
    return cache  # returns a dict: { "AAPL": <Series of date → price>, ... }

## 3. Helper — Looking up T and T+1 prices

For each news article, we need two prices:
- **T** — the closing price on the day the article was published
- **T+1** — the closing price on the *next available trading day*

"Next available trading day" means weekends and public holidays are skipped automatically (by yfinance), we just take the next date that has a price.

If there aren't at least two prices available on or after the article date (e.g. the article was published right before a long holiday or at the very end of the dataset), the function returns `None, None` and the row will be dropped later.

In [ ]:
def _get_t_and_t1(ticker, date_str, cache):
    # ticker: stock symbol e.g. "AAPL"
    # date_str: the article's publication date as a string e.g. "2021-03-15"
    # cache: the dictionary of prices returned by _fetch_price_cache

    prices = cache.get(ticker)  # look up this ticker's price series; returns None if missing
    if prices is None or prices.empty:
        return None, None  # no prices available for this ticker at all

    date = pd.Timestamp(date_str)  # convert the date string into a proper date object

    # get all dates in our price series that are on or after the article date
    # this automatically handles weekends and holidays
    # we are basically asking it to keep only the date of publication and the next one (using TRUE FALSE)
    future = prices.index[prices.index >= date]

    if len(future) < 2:
        # we need at least two trading days (T and T+1); if there aren't enough, skip
        return None, None

    # future[0] is T (the first available trading day on or after the article date)
    # future[1] is T+1 (the next trading day after that)
    return float(prices[future[0]]), float(prices[future[1]])

## 4. Helper — Assigning a label

Once we have the percentage change from T to T+1, this function converts it into one of three labels:

- **`up`** — price increased by more than the threshold (default: +1%)
- **`down`** — price decreased by more than the threshold (default: -1%)
- **`neutral`** — price moved less than ±1%, treated as no significant move

The threshold is configurable — Jack can pass a different value via the pipeline state if the team wants to experiment with stricter or looser definitions of "significant move" later on (keeping in mind of course that this would heavily affect our model's logic if we make the threshold for example too loose and put 0.5)

In [ ]:
def _assign_label(pct_change, threshold):
    # pct_change: the percentage change from T to T+1, e.g. 2.5 means +2.5%
    # threshold: the cutoff in decimal form, e.g. 0.01 means ±1%
    # threshold * 100 converts it to percentage form for comparison, e.g. 0.01 → 1.0

    if pct_change > threshold * 100:
        return "up"       # price rose more than the threshold
    elif pct_change < -threshold * 100:
        return "down"     # price fell more than the threshold
    return "neutral"      # price barely moved — inside the ±threshold band

## 5. Main processing node

This is the core of the agent (basically the function LangGraph calls when it's processing agent's turn in the pipeline. It runs all six steps in sequence and writes the final output file)

It receives the shared pipeline **state** (the dictionary passed between agents) and reads two things from it:
- `threshold` — the ±% cutoff for labelling (defaults to 1% if not set)
- `data_dir` — where to find the input files and write the output (defaults to the `data/` folder in the repo root)

### Step 1 — Load the raw news data
Reads `data/fnspid_raw.csv`, which contains all the financial news headlines with their publication dates and ticker symbols.

### Step 2 — Fetch or load the price cache
Downloading prices for hundreds of tickers is slow, so prices are cached in `data/price_cache.pkl` after the first run. On subsequent runs the cache is loaded from disk instead of hitting Yahoo Finance again.

### Step 3 — Assign prices and labels
For every row in the news data, looks up the T and T+1 prices using the helper above, computes the percentage change, and assigns an `up`/`down`/`neutral` label.

### Step 4 — Drop rows with no price data
Some rows can't be matched to prices (delisted tickers, edge-of-dataset dates, etc.). These are dropped entirely rather than passed on with missing values.

### Step 5 — Drop outliers
Rows where the percentage change is unusually large (defined by the standard IQR×1.5 statistical rule) are **removed** from the dataset entirely. We decided to drop them rather than flag them because keeping extreme price swings would skew FinBERT's training signal (should be around 1000 rows)

### Step 6 — Export
Writes `data/processed_data.csv` with the columns defined in `docs/data_contracts.md` (Handoff 1). Returns the output file path in the state so Nadi's Classifier Agent knows where to find it.

In [ ]:
def processing_node(state: PipelineState) -> dict:
    # state is a dictionary passed in by LangGraph containing settings for this run
    # -> dict means this function returns a dictionary (the updated state)

    threshold = state.get("threshold", 0.01)
    # default to the data/ folder inside the repo root
    # repo_root goes up two levels from agents/aurora_processing.py → repo root
    # then joins with "data" to get the data/ folder
    repo_root = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
    data_dir  = state.get("data_dir") or os.path.join(repo_root, "data")

    raw_path   = os.path.join(data_dir, "fnspid_raw.csv")    # input: data/fnspid_raw.csv
    cache_path = os.path.join(data_dir, "price_cache.pkl")   # cache: data/price_cache.pkl
    out_path   = os.path.join(data_dir, "processed_data.csv") # output: data/processed_data.csv

    # --- Step 1: Load raw news data ---
    if not os.path.exists(raw_path):
        raise FileNotFoundError(
            f"fnspid_raw.csv not found in {data_dir}. "
            "You could run Section 1 of processing_experiments.ipynb first if the file was deleted from repo"
        )
    print("[processing] Loading raw data ...")
    df = pd.read_csv(raw_path)
    print(f"  {len(df):,} rows, {df['ticker'].nunique()} tickers")

    # --- Step 2: Fetch or load price cache ---
    tickers = sorted(df["ticker"].unique().tolist())

    if os.path.exists(cache_path):
        print("[processing] Loading price cache ...")
        with open(cache_path, "rb") as f:
            price_cache = pickle.load(f)
        print(f"  {len(price_cache)} tickers in cache")
    else:
        date_min = pd.to_datetime(df["date"].min()) - pd.Timedelta(days=5)
        date_max = pd.to_datetime(df["date"].max()) + pd.Timedelta(days=10)
        print(f"[processing] Fetching prices for {len(tickers)} tickers ...")
        price_cache = _fetch_price_cache(tickers, date_min, date_max)
        with open(cache_path, "wb") as f:
            pickle.dump(price_cache, f)
        print(f"  Price cache saved to {cache_path}")

    # --- Step 3: Assign prices and labels ---
    print("[processing] Assigning prices and labels ...")
    results = [_get_t_and_t1(row["ticker"], row["date"], price_cache)
               for _, row in df.iterrows()]

    df["price_t"]  = [round(r[0], 2) if r[0] is not None else None for r in results]
    df["price_t1"] = [round(r[1], 2) if r[1] is not None else None for r in results]
    df["pct_change"] = [
        round((r[1] - r[0]) / r[0] * 100, 4) if r[0] and r[1] else None
        for r in results
    ]
    df["label"] = [
        _assign_label(p, threshold) if pd.notna(p) else None
        for p in df["pct_change"]
    ]

    # --- Step 4: Drop rows with no price data ---
    before = len(df)
    df = df.dropna(subset=["label"]).reset_index(drop=True)
    print(f"  Dropped {before - len(df):,} rows with no price data. Final: {len(df):,} rows")

    # --- Step 5: Drop outliers (IQR × 1.5) ---
    # IQR (interquartile range) is a standard way to detect unusually extreme values
    # q1 = 25th percentile, q3 = 75th percentile, iqr = the spread between them
    q1, q3 = df["pct_change"].quantile(0.25), df["pct_change"].quantile(0.75)
    iqr = q3 - q1
    # keep only rows inside [q1 - 1.5*iqr, q3 + 1.5*iqr] -> drop the rest entirely
    # (decided to drop rather than flag because extreme price swings skew FinBERT's training signal and the drop is only 6% of our data roughly)
    before_outliers = len(df)
    df = df[
        (df["pct_change"] >= q1 - 1.5 * iqr) &
        (df["pct_change"] <= q3 + 1.5 * iqr)
    ].reset_index(drop=True)
    print(f"  Outliers dropped: {before_outliers - len(df):,}. Final: {len(df):,} rows")

    # --- Step 6: Export ---
    # only include the columns defined in docs/data_contracts.md (Handoff 1)
    output_cols = [
        "article_id", "date", "ticker", "article_title",
        "price_t", "price_t1", "pct_change", "label",
    ]
    df[output_cols].to_csv(out_path, index=False)
    print(f"[processing] Written to {out_path}")

    label_counts = df["label"].value_counts().to_dict()
    print(f"  Labels: {label_counts}")

    return {"processed_data_path": out_path}

## 6. ProcessingAgent class

> **IMPORTANT as this is not python code I am used to since LangGraph is a new tool to me, read well for presentation and change if necessary**

This is the class that wraps `processing_node` and makes it a proper team-standard agent. Every agent in the project subclasses Jack's `Agent` base class from `agents/base.py` -> this is the team's agreed convention so that Jack's Manager Agent can call any agent the same way.

`processing_node` is the raw function LangGraph calls that does all the work, but to plug into the team's pipeline, each agent also needs to be a **class** that:
1. Builds its own LangGraph graph (via `build_graph`)
2. Exposes a single `.run()` method that Jack's Manager can call

### What `build_graph` does
Builds a simple graph: `START → processing → END`. It receives a `checkpointer` from the base class and passes it to `.compile()`. The checkpointer is what allows LangGraph to remember state across multiple `.run()` calls (e.g. if the pipeline loops).

### What `run` does
Takes any keyword arguments (like `threshold=0.01`), packages them as the initial state dictionary, and calls `self._invoke()` — a helper defined in the base class that runs the graph and returns the final state.

In [ ]:
class ProcessingAgent(Agent):
    # Agent is Jack's base class (subclassing it means we haveto implement build_graph and run)

    def build_graph(self, checkpointer):
        # build_graph is called once when ProcessingAgent() is created
        # it receives a checkpointer from the base class (MemorySaver by default)
        # and must return a compiled LangGraph graph
        builder = StateGraph(PipelineState)
        builder.add_node("processing", processing_node)  # register our node
        builder.add_edge(START, "processing")             # pipeline enters here
        builder.add_edge("processing", END)               # pipeline exits here
        return builder.compile(checkpointer=checkpointer) # checkpointer enables state memory across runs

    def run(self, **inputs) -> dict:
        # **inputs means any keyword arguments are collected into a dict
        # e.g. agent.run(threshold=0.01, data_dir="data/") → inputs = {"threshold": 0.01, "data_dir": "data/"}
        # self._invoke() is defined in the base class — it runs the graph and returns the final state
        return self._invoke(inputs)

## 7. Command-line entry point (TESTING)

This block only runs when the script is called directly from the terminal for testing, e.g.:

```bash
uv run agents/aurora_processing.py
uv run agents/aurora_processing.py --threshold 0.02
uv run agents/aurora_processing.py --data-dir /some/other/path
```

It accepts two optional arguments:
- `--threshold` — override the ±1% (`0.01`) default
- `--data-dir` — override the directory where input/output files are read and written

It uses `ProcessingAgent()` directly instead of building a graph manually so testing it from the terminal exercises the exact same code path Jack's Manager will use. (changed Jun 26th)

In [ ]:
# this block only runs when the file is executed directly (e.g. `uv run agents/aurora_processing.py`)
# it does NOT run when the file is imported by Jack's pipeline; that's what __name__ == "__main__" checks
if __name__ == "__main__":
    import argparse  # standard Python library for reading command-line arguments

    parser = argparse.ArgumentParser(
        description="Run the Processing Agent as a LangGraph node."
    )

    # optional argument: --threshold
    parser.add_argument("--threshold", type=float, default=0.01,
                        help="Price-change threshold in decimal form (default: 0.01 = ±1%)")

    # optional argument: --data-dir to override where files are read from / written to
    parser.add_argument("--data-dir", type=str, default=None,
                        help="Data directory containing fnspid_raw.csv (default: data/)")

    args = parser.parse_args()  # actually read the arguments from the terminal

    # create a ProcessingAgent instance -> this calls build_graph() internally via the base class
    # then call .run() with our arguments, which invokes the graph and returns the final state
    agent = ProcessingAgent()
    final_state = agent.run(
        threshold=args.threshold,
        data_dir=args.data_dir,
    )

    # print the path of the output file so we can confirm it was created
    print("\nOutput file:", final_state["processed_data_path"])